# Set-up

In [1]:
import os
import glob

import pandas as pd
import polars as pl
import snapatac2 as snap

from tqdm.auto import tqdm

The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
path_h5ads = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads"
path_cell_metadata = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/1_get_data/cell_metadata.tsv"
path_out = "/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data"

# Load and format cell meta

In [6]:
# Load cell metadata
cell_metadata = pd.read_csv(path_cell_metadata, sep="\t")
cell_metadata.head()

/tmp/ipykernel_2397660/2871173399.py:2: DtypeWarning: Columns (61,62,63,64,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  cell_metadata = pd.read_csv(path_cell_metadata, sep="\t")


,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,gex_barcode_cellranger,atac_barcode_cellranger,is_cell_cellranger,excluded_reason_cellranger,gex_raw_reads_cellranger,gex_mapped_reads_cellranger,...,reg.cca.clusters,seurat_clusters,sct.cca.clusters,reg.rpca.clusters,sct.rpca.clusters,reg.harmony.clusters,sct.harmony.clusters,mnn.clusters,rna_annotation,sample_name
0,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,2694,1778,CCAGCTAAGAGAGCCG-1,CGCTAACTCTAGGAAC-1,1,0,7833,7363,...,0,9,0,2,1,0,9,1,DE,HZ015_D4_rep1
1,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,2175,1404,GTCATTAAGTAAGAAC-1,CGTGTGTTCCTTCAAG-1,1,0,5790,5228,...,0,0,4,0,1,0,0,14,DE,HZ015_D4_rep1
2,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,1034,754,ATCCTGACAAGGTAAC-1,GATAACGGTTAGCTAC-1,1,0,2999,2754,...,0,1,2,0,1,0,1,5,DE,HZ015_D4_rep1
3,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,4914,2325,CAGCCAATCCGGTATG-1,GTCATTTAGTTAACGA-1,1,0,13873,12810,...,0,6,12,13,15,0,6,6,DE,HZ015_D4_rep1
4,rna_HZ015_D4_rep1_dual_modality_QCed_single_ti...,SeuratProject,5323,2408,CCTGTAACAGCTCATA-1,GTTAGGAGTTCAAGCT-1,1,0,15315,14466,...,0,3,7,2,5,0,3,4,DE,HZ015_D4_rep1


In [9]:
# Create barcode to cell id mapping
cell_metadata["label"] = cell_metadata['sample_name'] + "#" + cell_metadata['gex_barcode_cellranger']
cell_metadata['label'].value_counts()

label
HZ015_D4_rep1#CCAGCTAAGAGAGCCG-1     1
HZ019_D45_rep2#CCAGCCTGTTGTTCAC-1    1
HZ019_D45_rep2#GCCTGTGCAATCATGT-1    1
HZ019_D45_rep2#CCAACCAAGCCATCAG-1    1
HZ019_D45_rep2#GCGGGTTTCATAACTG-1    1
                                    ..
XW002_D12_rep1#GATTGCAGTTCCGGGA-1    1
XW002_D12_rep1#TTAGCAGGTTCATCTA-1    1
XW002_D12_rep1#GAAAGGCTCGCGACAC-1    1
XW002_D12_rep1#TCAATCGCAAAGCGGC-1    1
XW002_D9_rep1#GACAATACAGTTATCG-1     1
Name: count, Length: 104080, dtype: int64

In [15]:
cell_metadata["replicate"] = cell_metadata["sample_name"].str.split("_").str[-1]
cell_metadata["replicate"].value_counts()

replicate
rep1    61583
rep2    42497
Name: count, dtype: int64

In [25]:
cols_to_keep = [
    'label', 'sample_name', 'diff_batch',
    'diff_stage', 'replicate', 'rna_annotation', 
    'single_timepoint_seurat_annotation', 
    'D4_D7_continuous_seurat_annotation',
    'D7_D9_continuous_seurat_annotation',
    'D9_D12_continuous_seurat_annotation',
    'D22_D45_continuous_seurat_annotation',
    'D15_D22_continuous_seurat_annotation',
    'D12_D15_continuous_seurat_annotation',
]
index_col = "label"

In [26]:
cell_metadata_filt = cell_metadata[cols_to_keep].set_index(index_col)
cell_metadata_filt.head()

,sample_name,diff_batch,diff_stage,replicate,rna_annotation,single_timepoint_seurat_annotation,D4_D7_continuous_seurat_annotation,D7_D9_continuous_seurat_annotation,D9_D12_continuous_seurat_annotation,D22_D45_continuous_seurat_annotation,D15_D22_continuous_seurat_annotation,D12_D15_continuous_seurat_annotation
label,,,,,,,,,,,,
HZ015_D4_rep1#CCAGCTAAGAGAGCCG-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_ERBB4+,DE_ERBB4+,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#GTCATTAAGTAAGAAC-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_proliferating,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#ATCCTGACAAGGTAAC-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_MitoHi,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#CAGCCAATCCGGTATG-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_POLG2+,DE_GT_POLG2+,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#CCTGTAACAGCTCATA-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_proliferating,NaN,NaN,NaN,NaN,NaN


In [27]:
# rename rna_annotation to cell_type
cell_metadata_filt = cell_metadata_filt.rename(columns={"rna_annotation": "cell_type"})
cell_metadata_filt.head()

,sample_name,diff_batch,diff_stage,replicate,cell_type,single_timepoint_seurat_annotation,D4_D7_continuous_seurat_annotation,D7_D9_continuous_seurat_annotation,D9_D12_continuous_seurat_annotation,D22_D45_continuous_seurat_annotation,D15_D22_continuous_seurat_annotation,D12_D15_continuous_seurat_annotation
label,,,,,,,,,,,,
HZ015_D4_rep1#CCAGCTAAGAGAGCCG-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_ERBB4+,DE_ERBB4+,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#GTCATTAAGTAAGAAC-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_proliferating,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#ATCCTGACAAGGTAAC-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_MitoHi,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#CAGCCAATCCGGTATG-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_POLG2+,DE_GT_POLG2+,NaN,NaN,NaN,NaN,NaN
HZ015_D4_rep1#CCTGTAACAGCTCATA-1,HZ015_D4_rep1,HZ015,D4,rep1,DE,D4_DE_CDH8+,DE_GT_proliferating,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save updated cell metadata tp here: /cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data
cell_metadata_filt.to_csv(os.path.join(path_out, "subset", "cell_metadata.tsv"), sep="\t")

# Load AnnDataset

In [ ]:
dataset = snap.read_dataset(path_h5ads)
dataset

AnnDataSet object with n_obs x n_vars = 104080 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type'
    uns: 'AnnDataSet', 'macs3', 'reference_sequences'

In [29]:
dataset.obs["sample_name"] = cell_metadata_filt["sample_name"][dataset.obs_names].values.tolist()
dataset.obs["replicate"] = cell_metadata_filt["replicate"][dataset.obs_names].values.tolist()
dataset.obs["diff_batch"] = cell_metadata_filt["diff_batch"][dataset.obs_names].values.tolist()
dataset.obs["diff_stage"] = cell_metadata_filt["diff_stage"][dataset.obs_names].values.tolist()

In [32]:
dataset_obs = dataset.obs[:].to_pandas()

/tmp/ipykernel_2397660/4175217375.py:1: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  dataset_obs = dataset.obs[:].to_pandas()


In [34]:
pd.crosstab(dataset_obs["cell_type"], dataset_obs["replicate"])

replicate,rep1,rep2
cell_type,,
DE,6436,6694
ENP_phase1,154,1468
FB_FLT1,150,161
PFG1,2690,3
PFG2,8946,1218
PGT1,10361,53
PGT2,1806,89
PGT3,1279,374
PP1,2461,2601


In [36]:
dataset.close()

# All cell types and no reproducible peaks

## Call peaks with MACS3 per cell type

In [ ]:
if "macs3" not in dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=dataset,
        groupby="cell_type",
    )
    peaks = snap.tl.merge_peaks(dataset.uns['macs3'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = dataset.uns['macs3']

Calling peaks with MACS3...


2026-02-16 18:15:09 - INFO - Exporting fragments...
2026-02-16 18:39:08 - INFO - Calling peaks...
 59%|█████▉    | 13/22 [2:04:31<1:33:05, 620.57s/it]

In [ ]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in dataset.uns['macs3']:
    dataset.uns['macs3'][cell_id].write_csv(os.path.join(path_out, f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = dataset.uns['macs3'][cell_id].columns
    with open(os.path.join(path_out, f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

## Make a peak matrix

In [ ]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    dataset,
    peak_file=os.path.join(path_out, "consensus_peaks.bed"),
)

In [ ]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_matrix.h5ad"))

In [ ]:
dataset.close()

# All cell types and reproducible peaks

## Call peaks with MACS3 per cell type

In [3]:
dataset = snap.read_dataset(path_h5ads)
dataset

AnnDataSet object with n_obs x n_vars = 104080 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'macs3', 'AnnDataSet', 'reference_sequences'

In [ ]:
if "macs3_replicate" not in dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=dataset,
        groupby="cell_type",
        replicate="replicate",
        key_added="macs3_replicate"
    )
    peaks = snap.tl.merge_peaks(dataset.uns['macs3_replicate'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = dataset.uns['macs3_replicate']

Calling peaks with MACS3...


2026-02-23 08:58:52 - INFO - Exporting fragments...
2026-02-23 09:33:31 - INFO - Calling peaks...
2026-02-23 09:33:31 - INFO - Calling peaks...
  9%|▉         | 2/22 [59:48<9:55:42, 1787.12s/it] 

In [10]:
dataset

AnnDataSet object with n_obs x n_vars = 104080 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/subset/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'AnnDataSet', 'macs3', 'reference_sequences', 'macs3_replicate'

In [ ]:
dataset

In [9]:
print()

In [ ]:
dataset

In [13]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_calls", "all_replicate", "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "peak_calls", "all_replicate", "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in dataset.uns['macs3_replicate']:
    dataset.uns['macs3_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "all_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = dataset.uns['macs3_replicate'][cell_id].columns
    with open(os.path.join(path_out, "peak_calls", "all_replicate", f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

/tmp/ipykernel_2425615/1568424665.py:18: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  for cell_id in dataset.uns['macs3_replicate']:
/tmp/ipykernel_2425615/1568424665.py:19: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  dataset.uns['macs3_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "all_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
/tmp/ipykernel_2425615/1568424665.py:20: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  columns = dataset.uns['macs3_replicate'][cell_id].columns


## Make a peak matrix

In [14]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    dataset,
    peak_file=os.path.join(path_out, "peak_calls", "all_replicate", "consensus_peaks.bed")
)

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [15]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_calls", "all_replicate", "peak_matrix.h5ad"))

... storing 'sample' as categorical
... storing 'cell_type' as categorical
... storing 'cell_type' as categorical
... storing 'sample_name' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'diff_stage' as categorical


In [16]:
dataset.close()

# Endocrine cell types w/ reproducible peaks

In [17]:
ne_dataset = snap.read_dataset("/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/endocrine/_dataset.h5ads")
ne_dataset

AnnDataSet object with n_obs x n_vars = 47898 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'macs3_replicate', 'AnnDataSet', 'macs3', 'reference_sequences'

In [19]:
ne_dataset_obs = ne_dataset.obs[:].to_pandas()

/tmp/ipykernel_2425615/1296217026.py:1: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  ne_dataset_obs = ne_dataset.obs[:].to_pandas()


In [20]:
pd.crosstab(ne_dataset_obs["cell_type"], ne_dataset_obs["replicate"])

replicate,rep1,rep2
cell_type,,
ENP_phase1,154,1468
SC_delta_GHRL,404,452
early_ENP,3682,3890
early_SC_EC,4002,4439
early_SC_alpha,1477,1797
early_SC_beta,4702,6184
late_ENP,505,378
late_SC_EC,1972,2371
late_SC_alpha,2140,2490


## Call peaks with MACS3 per cell type

In [22]:
if "macs3_endocrine_replicate" not in ne_dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=ne_dataset,
        groupby="cell_type",
        replicate="replicate",
        key_added="macs3_endocrine_replicate"
    )
    peaks = snap.tl.merge_peaks(ne_dataset.uns['macs3_endocrine_replicate'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = ne_dataset.uns['macs3_endocrine_replicate']

2026-02-24 09:44:48 - INFO - Exporting fragments...
2026-02-24 09:54:58 - INFO - Calling peaks...


Calling peaks with MACS3...


100%|██████████| 11/11 [2:24:39<00:00, 789.06s/it]  
/tmp/ipykernel_2425615/1036867642.py:9: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  peaks = snap.tl.merge_peaks(ne_dataset.uns['macs3_endocrine_replicate'], snap.genome.hg38)


In [24]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_calls", "endocrine_replicate", "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "peak_calls", "endocrine_replicate", "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in ne_dataset.uns['macs3_endocrine_replicate']:
    ne_dataset.uns['macs3_endocrine_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "endocrine_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = ne_dataset.uns['macs3_endocrine_replicate'][cell_id].columns
    with open(os.path.join(path_out, "peak_calls", "endocrine_replicate", f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

/tmp/ipykernel_2425615/2240346850.py:18: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  for cell_id in ne_dataset.uns['macs3_endocrine_replicate']:
/tmp/ipykernel_2425615/2240346850.py:19: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  ne_dataset.uns['macs3_endocrine_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "endocrine_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
/tmp/ipykernel_2425615/2240346850.py:20: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  columns = ne_dataset.uns['macs3_endocrine_replicate'][cell_id].columns


## Make a peak matrix

In [25]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    ne_dataset,
    peak_file=os.path.join(path_out, "peak_calls", "endocrine_replicate", "consensus_peaks.bed")
)

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [26]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_calls", "endocrine_replicate", "peak_matrix.h5ad"))

... storing 'sample' as categorical
... storing 'cell_type' as categorical


... storing 'sample_name' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'diff_stage' as categorical


In [27]:
ne_dataset.close()

In [28]:
ne_dataset

Closed AnnDataSet object

# Endocrine cell types w/o reproducible peaks

In [29]:
endo_dataset = snap.read_dataset("/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/endocrine/_dataset.h5ads")
endo_dataset

AnnDataSet object with n_obs x n_vars = 47898 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'AnnDataSet', 'macs3_endocrine_replicate', 'macs3', 'macs3_replicate', 'reference_sequences'

In [30]:
endo_dataset_obs = endo_dataset.obs[:].to_pandas()
pd.crosstab(endo_dataset_obs["cell_type"], endo_dataset_obs["replicate"])

/tmp/ipykernel_2425615/3061940367.py:1: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  endo_dataset_obs = endo_dataset.obs[:].to_pandas()


replicate,rep1,rep2
cell_type,,
ENP_phase1,154,1468
SC_delta_GHRL,404,452
early_ENP,3682,3890
early_SC_EC,4002,4439
early_SC_alpha,1477,1797
early_SC_beta,4702,6184
late_ENP,505,378
late_SC_EC,1972,2371
late_SC_alpha,2140,2490


## Call peaks with MACS3 per cell type

In [31]:
if "macs3_endocrine" not in endo_dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=endo_dataset,
        groupby="cell_type",
        key_added="macs3_endocrine"
    )
    peaks = snap.tl.merge_peaks(endo_dataset.uns['macs3_endocrine'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = endo_dataset.uns['macs3_endocrine']

2026-02-24 14:08:45 - INFO - Exporting fragments...
2026-02-24 14:17:36 - INFO - Calling peaks...


Calling peaks with MACS3...


100%|██████████| 11/11 [1:06:49<00:00, 364.49s/it]
/tmp/ipykernel_2425615/3945659407.py:8: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  peaks = snap.tl.merge_peaks(endo_dataset.uns['macs3_endocrine'], snap.genome.hg38)


In [32]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_calls", "endocrine", "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "peak_calls", "endocrine", "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in endo_dataset.uns['macs3_endocrine']:
    endo_dataset.uns['macs3_endocrine'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "endocrine", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = endo_dataset.uns['macs3_endocrine'][cell_id].columns
    with open(os.path.join(path_out, "peak_calls", "endocrine", f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

/tmp/ipykernel_2425615/1984988068.py:18: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  for cell_id in endo_dataset.uns['macs3_endocrine']:
/tmp/ipykernel_2425615/1984988068.py:19: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  endo_dataset.uns['macs3_endocrine'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "endocrine", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
/tmp/ipykernel_2425615/1984988068.py:20: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  columns = endo_dataset.uns['macs3_endocrine'][cell_id].columns


## Make a peak matrix

In [33]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    endo_dataset,
    peak_file=os.path.join(path_out, "peak_calls", "endocrine", "consensus_peaks.bed")
)

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [34]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_calls", "endocrine", "peak_matrix.h5ad"))

... storing 'sample' as categorical
... storing 'cell_type' as categorical
... storing 'sample_name' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'diff_stage' as categorical


In [35]:
endo_dataset.close()

# Non-endocrine cell types w/ reproducible peaks

In [36]:
non_endo_dataset = snap.read_dataset("/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/non-endocrine/_dataset.h5ads")
non_endo_dataset

AnnDataSet object with n_obs x n_vars = 56182 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/non-endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'AnnDataSet', 'macs3_replicate', 'macs3', 'reference_sequences'

In [41]:
non_endo_dataset_obs = non_endo_dataset.obs[:].to_pandas()
pd.crosstab(non_endo_dataset_obs["cell_type"], non_endo_dataset_obs["replicate"])

/tmp/ipykernel_2425615/1092506745.py:1: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endo_dataset_obs = non_endo_dataset.obs[:].to_pandas()


replicate,rep1,rep2
cell_type,,
DE,6436,6694
FB_FLT1,150,161
PFG1,2690,3
PFG2,8946,1218
PGT1,10361,53
PGT2,1806,89
PGT3,1279,374
PP1,2461,2601
PP2,3615,3174


## Call peaks with MACS3 per cell type

In [46]:
if "macs3_non_endocrine_replicate" not in non_endo_dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=non_endo_dataset,
        groupby="cell_type",
        replicate="replicate",
        key_added="macs3_non_endocrine_replicate"
    )
    peaks = snap.tl.merge_peaks(non_endo_dataset.uns['macs3_non_endocrine_replicate'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = snap.tl.merge_peaks(non_endo_dataset.uns['macs3_non_endocrine_replicate'], snap.genome.hg38)
    #peaks = non_endo_dataset.uns['macs3_non_endocrine_replicate']

Peaks already called.


/tmp/ipykernel_2425615/3299615057.py:12: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  peaks = snap.tl.merge_peaks(non_endo_dataset.uns['macs3_non_endocrine_replicate'], snap.genome.hg38)


In [47]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_calls", "non-endocrine_replicate", "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "peak_calls", "non-endocrine_replicate", "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in non_endo_dataset.uns['macs3_non_endocrine_replicate']:
    non_endo_dataset.uns['macs3_non_endocrine_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "non-endocrine_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = non_endo_dataset.uns['macs3_non_endocrine_replicate'][cell_id].columns
    with open(os.path.join(path_out, "peak_calls", "non-endocrine_replicate", f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

/tmp/ipykernel_2425615/844376609.py:18: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  for cell_id in non_endo_dataset.uns['macs3_non_endocrine_replicate']:
/tmp/ipykernel_2425615/844376609.py:19: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endo_dataset.uns['macs3_non_endocrine_replicate'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "non-endocrine_replicate", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
/tmp/ipykernel_2425615/844376609.py:20: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  columns = non_endo_dataset.uns['macs3_non_endocrine_replicate'][cell_id].columns


## Make a peak matrix

In [48]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    non_endo_dataset,
    peak_file=os.path.join(path_out, "peak_calls", "non-endocrine_replicate", "consensus_peaks.bed")
)

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [49]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_calls", "non-endocrine_replicate", "peak_matrix.h5ad"))

... storing 'sample' as categorical
... storing 'cell_type' as categorical
... storing 'sample_name' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'diff_stage' as categorical


In [50]:
non_endo_dataset.close()

# Non-endocrine cell types w/o reproducible peaks

In [51]:
non_endo_dataset = snap.read_dataset("/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/non-endocrine/_dataset.h5ads")
non_endo_dataset

AnnDataSet object with n_obs x n_vars = 56182 x 0 backed at '/cellar/users/aklie/data/datasets/sc-islet-differentiation_10X-Multiome/results/2_process_data/non-endocrine/_dataset.h5ads'
contains 14 AnnData objects with keys: 'HZ015_D4_rep1', 'HZ015_D4_rep2', 'HZ015_D7_rep1', 'HZ015_D9_rep1', 'HZ019_D45_rep1', 'HZ019_D45_rep2', 'HZ021_D15_rep1', 'HZ021_D15_rep2', 'HZ021_D22_rep1', 'HZ021_D22_rep2', 'XW002_D12_rep1', 'XW002_D12_rep2', 'XW002_D7_rep1', 'XW002_D9_rep1'
    obs: 'sample', 'cell_type', 'sample_name', 'replicate', 'diff_batch', 'diff_stage'
    uns: 'macs3', 'AnnDataSet', 'macs3_non_endocrine_replicate', 'macs3_replicate', 'reference_sequences'

In [52]:
non_endo_dataset_obs = non_endo_dataset.obs[:].to_pandas()
pd.crosstab(non_endo_dataset_obs["cell_type"], non_endo_dataset_obs["replicate"])

/tmp/ipykernel_2425615/1092506745.py:1: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endo_dataset_obs = non_endo_dataset.obs[:].to_pandas()


replicate,rep1,rep2
cell_type,,
DE,6436,6694
FB_FLT1,150,161
PFG1,2690,3
PFG2,8946,1218
PGT1,10361,53
PGT2,1806,89
PGT3,1279,374
PP1,2461,2601
PP2,3615,3174


## Call peaks with MACS3 per cell type

In [53]:
if "macs3_non_endocrine" not in non_endo_dataset.uns:
    print("Calling peaks with MACS3...")
    snap.tl.macs3(
        adata=non_endo_dataset,
        groupby="cell_type",
        key_added="macs3_non_endocrine"
    )
    peaks = snap.tl.merge_peaks(non_endo_dataset.uns['macs3_non_endocrine'], snap.genome.hg38)
else:
    print("Peaks already called.")
    peaks = non_endo_dataset.uns['macs3_non_endocrine']

2026-02-25 08:40:33 - INFO - Exporting fragments...


Calling peaks with MACS3...


2026-02-25 08:56:59 - INFO - Calling peaks...
100%|██████████| 11/11 [1:58:54<00:00, 648.57s/it] 
/tmp/ipykernel_2425615/2173785365.py:8: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  peaks = snap.tl.merge_peaks(non_endo_dataset.uns['macs3_non_endocrine'], snap.genome.hg38)


In [54]:
# Write out the sources of each peak
peaks.to_pandas().to_csv(os.path.join(path_out, "peak_calls", "non-endocrine", "peak_sources.tsv"), sep="\t", index=False, header=True)

# Write out the consensus peak set
peak_ids = peaks.with_columns([
    pl.col('Peaks').str.split(":").list.get(0).alias('chrom'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(0).cast(pl.Int64).alias('start'),
    pl.col('Peaks').str.split(":").list.get(1).str.split("-").list.get(1).cast(pl.Int64).alias('end')
]).select(['chrom', 'start', 'end']).to_pandas()
peak_ids.to_csv(
    os.path.join(path_out, "peak_calls", "non-endocrine", "consensus_peaks.bed"),
    sep="\t",
    header=False,
    index=False,
)

# Write out BED and narrowPeak files
for cell_id in non_endo_dataset.uns['macs3_non_endocrine']:
    non_endo_dataset.uns['macs3_non_endocrine'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "non-endocrine", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
    columns = non_endo_dataset.uns['macs3_non_endocrine'][cell_id].columns
    with open(os.path.join(path_out, "peak_calls", "non-endocrine", f"{cell_id}.narrowPeak.schema"), "w") as f:
        f.write("\n".join(columns))

/tmp/ipykernel_2425615/810400863.py:18: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  for cell_id in non_endo_dataset.uns['macs3_non_endocrine']:
/tmp/ipykernel_2425615/810400863.py:19: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  non_endo_dataset.uns['macs3_non_endocrine'][cell_id].write_csv(os.path.join(path_out, "peak_calls", "non-endocrine", f"{cell_id}.narrowPeak"), separator="\t", include_header=False)
/tmp/ipykernel_2425615/810400863.py:20: DeprecationWarning: `_import_from_c` is deprecated; use `_import_arrow_from_c` instead. If you are using an extension, please compile it with the latest 'pyo3-polars'
  columns = non_endo_dataset.uns['macs3_non_endocrine'][cell_id].columns


## Make a peak matrix

In [55]:
# make peak matrix
peak_matrix = snap.pp.make_peak_matrix(
    non_endo_dataset,
    peak_file=os.path.join(path_out, "peak_calls", "non-endocrine", "consensus_peaks.bed")
)

/cellar/users/aklie/opt/miniconda3/envs/scverse-lite-py311/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [56]:
peak_matrix.write_h5ad(os.path.join(path_out, "peak_calls", "non-endocrine", "peak_matrix.h5ad"))

... storing 'sample' as categorical
... storing 'cell_type' as categorical
... storing 'cell_type' as categorical
... storing 'sample_name' as categorical
... storing 'replicate' as categorical
... storing 'diff_batch' as categorical
... storing 'diff_stage' as categorical


In [57]:
non_endo_dataset.close()

# DONE!

---